In [1]:
import torch
print("File    :", torch.__file__)
print("Version :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
    print("VRAM    :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

File    : d:\End-to-end-ML\Customer-Risk-Escalation-Engine\venv\Lib\site-packages\torch\__init__.py
Version : 2.5.1+cu121
CUDA    : True
GPU     : NVIDIA GeForce RTX 3050 Laptop GPU
VRAM    : 4.0 GB


In [2]:
import os
import numpy as np
import pandas as pd
import torch
from transformers import DistilBertTokenizer, DistilBertModel
from torch.utils.data import Dataset, DataLoader
import mlflow
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

os.chdir(r'D:\End-to-end-ML\Customer-Risk-Escalation-Engine')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

d:\End-to-end-ML\Customer-Risk-Escalation-Engine\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
text_train = pd.read_csv('data/processed_data/text_train.csv').squeeze()
text_test  = pd.read_csv('data/processed_data/text_test.csv').squeeze()

In [ ]:
print(f"text_train : {text_train.shape}") # type: ignore
print(f"text_test  : {text_test.shape}")  # type: ignore

text_train : (80000, 2)
text_test  : (20000, 2)


In [6]:
MODEL_NAME = 'distilbert-base-uncased'

print("Loading tokenizer...")
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")
model = DistilBertModel.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()  # Inference mode — no training happening

print(f"\nModel loaded on : {device} ✅")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading tokenizer...


Loading model...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4381.21it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Model loaded on : cuda ✅
Model parameters: 66,362,880


In [7]:
class TicketTextDataset(Dataset):
    """
    Converts raw text strings into tokenized input
    that DistilBERT can understand.
    """
    def __init__(self, texts, tokenizer, max_length=128):
        self.texts     = texts.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])

        encoding = self.tokenizer(
            text,
            max_length      = self.max_length,
            padding         = 'max_length',
            truncation      = True,
            return_tensors  = 'pt'
        )

        return {
            'input_ids'      : encoding['input_ids'].squeeze(),
            'attention_mask' : encoding['attention_mask'].squeeze()
        }

print("Dataset class defined ✅")

Dataset class defined ✅
